In [1]:
# [1] 설치
# 최초 1회만 터미널에서 실행
# pip install finance-datareader yfinance pandas numpy requests lxml beautifulsoup4 html5lib

from datetime import datetime
from pathlib import Path
from io import StringIO
import time
import re

import numpy as np
import pandas as pd
import yfinance as yf
import FinanceDataReader as fdr
import requests


# [2] 기본 설정

START_DATE = "2020-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

OUT_DIR = Path("./market_agent_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PRICE = OUT_DIR / "market_price_data.csv"
OUT_FLOW = OUT_DIR / "market_flow_data.csv"

KR_STOCKS = {
    "005930": "Samsung Electronics",
    "000660": "SK Hynix",
    "042700": "Hanmi Semiconductor",
}

KR_INDEXES = {
    "KS11": "KOSPI",
    "KQ11": "KOSDAQ",
}

GLOBAL_ASSETS = {
    "SOXX": ("iShares Semiconductor ETF", "US_ETF"),
    "SMH": ("VanEck Semiconductor ETF", "US_ETF"),
    "^IXIC": ("NASDAQ Composite", "US_INDEX"),
    "^GSPC": ("S&P 500", "US_INDEX"),
    "KRW=X": ("USD/KRW", "FX"),
}


# [3] 공통 정리 함수

def clean_number(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.replace(",", "")
    x = x.replace("%", "")
    x = x.replace("+", "")
    x = x.replace("−", "-")
    x = x.replace("▲", "")
    x = x.replace("▼", "-")
    x = re.sub(r"[^0-9.\-]", "", x)

    if x in ["", "-", "."]:
        return np.nan

    try:
        return float(x)
    except Exception:
        return np.nan


def flatten_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x).strip() for x in col if str(x) != "nan"])
            for col in df.columns
        ]
    else:
        df.columns = [str(c).strip() for c in df.columns]

    return df


def find_col(columns, must_include=None, exclude=None):
    must_include = must_include or []
    exclude = exclude or []

    for col in columns:
        col_str = str(col).strip()

        if all(key in col_str for key in must_include) and not any(key in col_str for key in exclude):
            return col

    return None


def to_long_price_df(df, asset_code, asset_name, market, source):
    columns = [
        "Date", "AssetCode", "AssetName", "Market", "Source",
        "Open", "High", "Low", "Close", "Volume", "Value"
    ]

    if df is None or df.empty:
        return pd.DataFrame(columns=columns)

    out = df.copy().reset_index()

    if "Date" not in out.columns:
        out = out.rename(columns={out.columns[0]: "Date"})

    for col in ["Open", "High", "Low", "Close", "Volume"]:
        if col not in out.columns:
            out[col] = np.nan

    if "Value" not in out.columns:
        out["Value"] = np.nan

    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out["AssetCode"] = asset_code
    out["AssetName"] = asset_name
    out["Market"] = market
    out["Source"] = source

    out = out[columns].copy()
    out = out.dropna(subset=["Date"])
    out = out.sort_values("Date").reset_index(drop=True)

    return out


# [4] 국내 주식 가격 수집

def fetch_fdr_kr_stock_prices(kr_stocks):
    frames = []

    for ticker, asset_name in kr_stocks.items():
        try:
            raw = fdr.DataReader(ticker, START_DATE, END_DATE)

            price_df = to_long_price_df(
                df=raw,
                asset_code=ticker,
                asset_name=asset_name,
                market="KR_STOCK",
                source="FinanceDataReader"
            )

            if not price_df.empty:
                frames.append(price_df)
                print(f"[OK] FDR stock: {ticker}, rows={len(price_df)}")
            else:
                print(f"[WARN] FDR stock 빈 데이터: {ticker}")

        except Exception as e:
            print(f"[ERROR] FDR stock 실패: {ticker} / {e}")

        time.sleep(0.2)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# [5] 국내 지수 가격 수집

def fetch_fdr_kr_index_prices(kr_indexes):
    frames = []

    for code, name in kr_indexes.items():
        try:
            raw = fdr.DataReader(code, START_DATE, END_DATE)

            price_df = to_long_price_df(
                df=raw,
                asset_code=code,
                asset_name=name,
                market="KR_INDEX",
                source="FinanceDataReader"
            )

            if not price_df.empty:
                frames.append(price_df)
                print(f"[OK] FDR index: {code}, rows={len(price_df)}")
            else:
                print(f"[WARN] FDR index 빈 데이터: {code}")

        except Exception as e:
            print(f"[ERROR] FDR index 실패: {code} / {e}")

        time.sleep(0.2)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# [6] 해외 가격 수집

def fetch_yfinance_global_prices(global_assets):
    frames = []

    for ticker, (asset_name, market) in global_assets.items():
        try:
            raw = yf.download(
                ticker,
                start=START_DATE,
                end=END_DATE,
                auto_adjust=False,
                progress=False
            )

            if raw is None or raw.empty:
                print(f"[WARN] yfinance 빈 데이터: {ticker}")
                continue

            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.get_level_values(0)

            keep_cols = [
                c for c in ["Open", "High", "Low", "Close", "Volume"]
                if c in raw.columns
            ]
            raw = raw[keep_cols].copy()

            price_df = to_long_price_df(
                df=raw,
                asset_code=ticker,
                asset_name=asset_name,
                market=market,
                source="yfinance"
            )

            if not price_df.empty:
                frames.append(price_df)
                print(f"[OK] yfinance: {ticker}, rows={len(price_df)}")
            else:
                print(f"[WARN] yfinance 변환 후 빈 데이터: {ticker}")

        except Exception as e:
            print(f"[ERROR] yfinance 실패: {ticker} / {e}")

        time.sleep(0.2)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# [7] 네이버 금융 외국인/기관 일별 매매동향 수집
# NetBuyVolume: 순매수 수량 기준

def fetch_naver_flow_one_stock(ticker, asset_name, max_pages=250, debug_first_page=True):
    columns = [
        "Date", "AssetCode", "AssetName", "Market", "Source",
        "InvestorType", "NetBuyVolume"
    ]

    frames = []
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    for page in range(1, max_pages + 1):
        url = f"https://finance.naver.com/item/frgn.naver?code={ticker}&page={page}"

        try:
            res = requests.get(url, headers=headers, timeout=10)
            res.raise_for_status()

            tables = pd.read_html(StringIO(res.text), encoding="cp949")

            target = None
            date_col = None
            foreign_col = None
            institution_col = None

            if debug_first_page and page == 1:
                print(f"\n[DEBUG] Naver tables for {ticker}, page={page}, n_tables={len(tables)}")
                for i, t in enumerate(tables):
                    t = flatten_columns(t.copy())
                    print(f"[DEBUG] table={i}, columns={list(t.columns)}")

            for table in tables:
                table = flatten_columns(table.copy())
                table = table.dropna(how="all")

                if table.empty:
                    continue

                cols = list(table.columns)

                date_candidate = find_col(cols, must_include=["날짜"])
                foreign_candidate = find_col(
                    cols,
                    must_include=["외국인"],
                    exclude=["보유", "비율", "소진"]
                )
                institution_candidate = find_col(
                    cols,
                    must_include=["기관"],
                    exclude=["보유", "비율", "소진"]
                )

                if date_candidate is not None and (
                    foreign_candidate is not None or institution_candidate is not None
                ):
                    target = table.copy()
                    date_col = date_candidate
                    foreign_col = foreign_candidate
                    institution_col = institution_candidate
                    break

            if target is None or target.empty:
                continue

            target["Date"] = pd.to_datetime(target[date_col], errors="coerce")
            target = target.dropna(subset=["Date"])

            if target.empty:
                continue

            oldest_on_page = target["Date"].min()

            target = target[target["Date"] >= pd.to_datetime(START_DATE)]
            target = target[target["Date"] <= pd.to_datetime(END_DATE)]

            if target.empty:
                if oldest_on_page < pd.to_datetime(START_DATE):
                    break
                continue

            page_rows = []

            if foreign_col is not None:
                temp = target[["Date", foreign_col]].copy()
                temp["InvestorType"] = "Foreign"
                temp["NetBuyVolume"] = temp[foreign_col].apply(clean_number)
                page_rows.append(temp[["Date", "InvestorType", "NetBuyVolume"]])

            if institution_col is not None:
                temp = target[["Date", institution_col]].copy()
                temp["InvestorType"] = "Institution"
                temp["NetBuyVolume"] = temp[institution_col].apply(clean_number)
                page_rows.append(temp[["Date", "InvestorType", "NetBuyVolume"]])

            if page_rows:
                page_df = pd.concat(page_rows, ignore_index=True)
                page_df["AssetCode"] = ticker
                page_df["AssetName"] = asset_name
                page_df["Market"] = "KR_STOCK"
                page_df["Source"] = "NaverFinance"
                page_df = page_df[columns]
                frames.append(page_df)

        except Exception as e:
            print(f"[WARN] Naver flow page 실패: {ticker}, page={page}, error={e}")

        time.sleep(0.15)

    if not frames:
        print(f"[WARN] Naver flow 빈 데이터: {ticker}")
        return pd.DataFrame(columns=columns)

    result = pd.concat(frames, ignore_index=True)
    result = result.dropna(subset=["Date"])
    result = result.dropna(subset=["NetBuyVolume"])
    result = result.drop_duplicates(subset=["Date", "AssetCode", "InvestorType"])
    result = result.sort_values(["Date", "AssetCode", "InvestorType"]).reset_index(drop=True)

    print(f"[OK] Naver flow: {ticker}, rows={len(result)}")
    return result


def fetch_naver_flows(kr_stocks, max_pages=250):
    columns = [
        "Date", "AssetCode", "AssetName", "Market", "Source",
        "InvestorType", "NetBuyVolume"
    ]

    frames = []

    for ticker, asset_name in kr_stocks.items():
        flow_df = fetch_naver_flow_one_stock(
            ticker=ticker,
            asset_name=asset_name,
            max_pages=max_pages,
            debug_first_page=True
        )

        if not flow_df.empty:
            frames.append(flow_df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)


# [8] 실행

print("[START] Market Agent data collection")

kr_stock_price_df = fetch_fdr_kr_stock_prices(KR_STOCKS)
kr_index_price_df = fetch_fdr_kr_index_prices(KR_INDEXES)
global_price_df = fetch_yfinance_global_prices(GLOBAL_ASSETS)
flow_df = fetch_naver_flows(KR_STOCKS, max_pages=250)

price_df = pd.concat(
    [kr_stock_price_df, kr_index_price_df, global_price_df],
    ignore_index=True
)

price_df["Date"] = pd.to_datetime(price_df["Date"], errors="coerce")
price_df = price_df.dropna(subset=["Date"])
price_df = price_df.sort_values(["Date", "AssetCode"]).drop_duplicates().reset_index(drop=True)

flow_df["Date"] = pd.to_datetime(flow_df["Date"], errors="coerce")
flow_df = flow_df.dropna(subset=["Date"])
flow_df = flow_df.sort_values(["Date", "AssetCode", "InvestorType"]).drop_duplicates().reset_index(drop=True)

price_df.to_csv(OUT_PRICE, index=False, encoding="utf-8-sig")
flow_df.to_csv(OUT_FLOW, index=False, encoding="utf-8-sig")

print("[DONE] Saved files")
print("price:", OUT_PRICE, price_df.shape)
print("flow :", OUT_FLOW, flow_df.shape)


# [9] 결과 점검

print("\n[PRICE COUNT]")
if not price_df.empty:
    price_count = (
        price_df.groupby(["AssetCode", "AssetName", "Market", "Source"])
        .size()
        .reset_index(name="RowCount")
        .sort_values(["Market", "AssetCode"])
        .reset_index(drop=True)
    )
    display(price_count)
else:
    print("price_df is empty")

print("\n[FLOW COUNT]")
if not flow_df.empty:
    flow_count = (
        flow_df.groupby(["AssetCode", "AssetName", "InvestorType"])
        .size()
        .reset_index(name="RowCount")
        .sort_values(["AssetCode", "InvestorType"])
        .reset_index(drop=True)
    )
    display(flow_count)
else:
    print("flow_df is empty")

print("\n[PRICE HEAD]")
display(price_df.head(10))

print("\n[FLOW HEAD]")
display(flow_df.head(10))

[START] Market Agent data collection
[OK] FDR stock: 005930, rows=1550
[OK] FDR stock: 000660, rows=1550
[OK] FDR stock: 042700, rows=1550
[OK] FDR index: KS11, rows=1550
[OK] FDR index: KQ11, rows=1550
[OK] yfinance: SOXX, rows=1586
[OK] yfinance: SMH, rows=1586
[OK] yfinance: ^IXIC, rows=1586
[OK] yfinance: ^GSPC, rows=1586
[OK] yfinance: KRW=X, rows=1644

[DEBUG] Naver tables for 005930, page=1, n_tables=13
[DEBUG] table=0, columns=['0', '1', '2']
[DEBUG] table=1, columns=['0', '1', '2']
[DEBUG] table=2, columns=['매도상위', '거래량', '매수상위', '거래량.1']
[DEBUG] table=3, columns=['날짜_날짜', '종가_종가', '전일비_전일비', '등락률_등락률', '거래량_거래량', '기관_순매매량', '외국인_순매매량', '외국인_보유주수', '외국인_보유율']
[DEBUG] table=4, columns=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']
[DEBUG] table=5, columns=['0', '1']
[DEBUG] table=6, columns=['0', '1']
[DEBUG] table=7, columns=['0', '1']
[DEBUG] table=8, columns=['0', '1']
[DEBUG] table=9, columns=['0', '1']
[DEBUG] table=10, columns=['매도잔량', '호가(20분지연)', '매수잔량']

,AssetCode,AssetName,Market,Source,RowCount
0,KRW=X,USD/KRW,FX,yfinance,1644
1,KQ11,KOSDAQ,KR_INDEX,FinanceDataReader,1550
2,KS11,KOSPI,KR_INDEX,FinanceDataReader,1550
3,000660,SK Hynix,KR_STOCK,FinanceDataReader,1550
4,005930,Samsung Electronics,KR_STOCK,FinanceDataReader,1550
5,042700,Hanmi Semiconductor,KR_STOCK,FinanceDataReader,1550
6,SMH,VanEck Semiconductor ETF,US_ETF,yfinance,1586
7,SOXX,iShares Semiconductor ETF,US_ETF,yfinance,1586
8,^GSPC,S&P 500,US_INDEX,yfinance,1586
9,^IXIC,NASDAQ Composite,US_INDEX,yfinance,1586



[FLOW COUNT]


,AssetCode,AssetName,InvestorType,RowCount
0,000660,SK Hynix,Foreign,1550
1,000660,SK Hynix,Institution,1550
2,005930,Samsung Electronics,Foreign,1550
3,005930,Samsung Electronics,Institution,1550
4,042700,Hanmi Semiconductor,Foreign,1550
5,042700,Hanmi Semiconductor,Institution,1550



[PRICE HEAD]


,Date,AssetCode,AssetName,Market,Source,Open,High,Low,Close,Volume,Value
0,2020-01-01,KRW=X,USD/KRW,FX,yfinance,1154.400024,1154.400024,1145.449951,1153.750000,0,NaN
1,2020-01-02,000660,SK Hynix,KR_STOCK,FinanceDataReader,96000.000000,96200.000000,94100.000000,94700.000000,2342070,NaN
2,2020-01-02,005930,Samsung Electronics,KR_STOCK,FinanceDataReader,55500.000000,56000.000000,55000.000000,55200.000000,12993228,NaN
3,2020-01-02,042700,Hanmi Semiconductor,KR_STOCK,FinanceDataReader,4055.000000,4125.000000,3970.000000,3980.000000,366562,NaN
4,2020-01-02,KQ11,KOSDAQ,KR_INDEX,FinanceDataReader,672.530000,674.300000,666.620000,674.020000,783730760,NaN
5,2020-01-02,KRW=X,USD/KRW,FX,yfinance,1153.959961,1160.189941,1145.199951,1153.969971,0,NaN
6,2020-01-02,KS11,KOSPI,KR_INDEX,FinanceDataReader,2201.210000,2202.320000,2171.840000,2175.170000,494677752,NaN
7,2020-01-02,SMH,VanEck Semiconductor ETF,US_ETF,yfinance,71.894997,72.470001,71.654999,72.339996,5200400,NaN
8,2020-01-02,SOXX,iShares Semiconductor ETF,US_ETF,yfinance,84.753334,85.430000,84.333336,85.430000,1275300,NaN
9,2020-01-02,^GSPC,S&P 500,US_INDEX,yfinance,3244.669922,3258.139893,3235.530029,3257.850098,3459930000,NaN



[FLOW HEAD]


,Date,AssetCode,AssetName,Market,Source,InvestorType,NetBuyVolume
0,2020-01-02,000660,SK Hynix,KR_STOCK,NaverFinance,Foreign,558612.0
1,2020-01-02,000660,SK Hynix,KR_STOCK,NaverFinance,Institution,-690924.0
2,2020-01-02,005930,Samsung Electronics,KR_STOCK,NaverFinance,Foreign,-528398.0
3,2020-01-02,005930,Samsung Electronics,KR_STOCK,NaverFinance,Institution,-2354766.0
4,2020-01-02,042700,Hanmi Semiconductor,KR_STOCK,NaverFinance,Foreign,-15903.0
5,2020-01-02,042700,Hanmi Semiconductor,KR_STOCK,NaverFinance,Institution,7510.0
6,2020-01-03,000660,SK Hynix,KR_STOCK,NaverFinance,Foreign,-165735.0
7,2020-01-03,000660,SK Hynix,KR_STOCK,NaverFinance,Institution,-98745.0
8,2020-01-03,005930,Samsung Electronics,KR_STOCK,NaverFinance,Foreign,1120131.0
9,2020-01-03,005930,Samsung Electronics,KR_STOCK,NaverFinance,Institution,-2228329.0


In [3]:
# [10] Feature Engineering Layer
# price_data + flow_data를 이용해 Market Agent 분석용 feature 생성

from pathlib import Path
import numpy as np
import pandas as pd

OUT_DIR = Path("./market_agent_data")
OUT_FEATURE = OUT_DIR / "market_feature_data.csv"

PRICE_FILE = OUT_DIR / "market_price_data.csv"
FLOW_FILE = OUT_DIR / "market_flow_data.csv"

if "price_df" not in globals():
    price_df = pd.read_csv(PRICE_FILE)

if "flow_df" not in globals():
    flow_df = pd.read_csv(FLOW_FILE)

price_df["Date"] = pd.to_datetime(price_df["Date"], errors="coerce")
flow_df["Date"] = pd.to_datetime(flow_df["Date"], errors="coerce")

price_df = price_df.dropna(subset=["Date"]).copy()
flow_df = flow_df.dropna(subset=["Date"]).copy()

price_df = price_df.sort_values(["AssetCode", "Date"]).reset_index(drop=True)
flow_df = flow_df.sort_values(["AssetCode", "InvestorType", "Date"]).reset_index(drop=True)


def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))

    return rsi


def add_price_features(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("Date").copy()

    close = group["Close"]
    volume = group["Volume"]

    group["Return_1D"] = close.pct_change(1)
    group["Return_5D"] = close.pct_change(5)
    group["Return_20D"] = close.pct_change(20)
    group["Return_60D"] = close.pct_change(60)
    group["Return_120D"] = close.pct_change(120)
    group["Return_240D"] = close.pct_change(240)

    group["MA30"] = close.rolling(30).mean()
    group["MA60"] = close.rolling(60).mean()
    group["MA120"] = close.rolling(120).mean()

    group["MA30_Above_MA60"] = (group["MA30"] > group["MA60"]).astype(int)
    group["MA60_Above_MA120"] = (group["MA60"] > group["MA120"]).astype(int)
    group["MA_Alignment"] = (
        (group["MA30"] > group["MA60"]) &
        (group["MA60"] > group["MA120"])
    ).astype(int)

    group["Close_to_MA30"] = close / group["MA30"] - 1
    group["Close_to_MA60"] = close / group["MA60"] - 1
    group["Close_to_MA120"] = close / group["MA120"] - 1

    group["High_52W"] = close.rolling(240).max()
    group["Drawdown_52W"] = close / group["High_52W"] - 1

    group["Volatility_20D"] = group["Return_1D"].rolling(20).std()
    group["Volatility_60D"] = group["Return_1D"].rolling(60).std()

    group["RSI14"] = compute_rsi(close, window=14)

    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()

    group["MACD"] = ema12 - ema26
    group["MACD_Signal"] = group["MACD"].ewm(span=9, adjust=False).mean()
    group["MACD_Hist"] = group["MACD"] - group["MACD_Signal"]

    group["MACD_GoldenCross"] = (
        (group["MACD"] > group["MACD_Signal"]) &
        (group["MACD"].shift(1) <= group["MACD_Signal"].shift(1))
    ).astype(int)

    group["ROC60"] = close / close.shift(60) - 1

    group["Volume_MA20"] = volume.rolling(20).mean()
    group["Volume_Ratio_20D"] = volume / group["Volume_MA20"]

    return group


price_feature_df = (
    price_df
    .groupby("AssetCode", group_keys=False)
    .apply(add_price_features)
    .reset_index(drop=True)
)


flow_wide = (
    flow_df
    .pivot_table(
        index=["Date", "AssetCode", "AssetName", "Market", "Source"],
        columns="InvestorType",
        values="NetBuyVolume",
        aggfunc="sum"
    )
    .reset_index()
)

flow_wide.columns.name = None

for col in ["Foreign", "Institution"]:
    if col not in flow_wide.columns:
        flow_wide[col] = 0.0

flow_wide = flow_wide.sort_values(["AssetCode", "Date"]).reset_index(drop=True)


def add_flow_features(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("Date").copy()

    group["Foreign_5D"] = group["Foreign"].rolling(5).sum()
    group["Foreign_20D"] = group["Foreign"].rolling(20).sum()
    group["Institution_5D"] = group["Institution"].rolling(5).sum()
    group["Institution_20D"] = group["Institution"].rolling(20).sum()

    group["TotalFlow"] = group["Foreign"] + group["Institution"]
    group["TotalFlow_5D"] = group["TotalFlow"].rolling(5).sum()
    group["TotalFlow_20D"] = group["TotalFlow"].rolling(20).sum()

    group["Foreign_Positive_20D"] = (group["Foreign_20D"] > 0).astype(int)
    group["Institution_Positive_20D"] = (group["Institution_20D"] > 0).astype(int)

    group["Flow_Confirm_20D"] = (
        (group["Foreign_20D"] > 0) &
        (group["Institution_20D"] > 0)
    ).astype(int)

    return group


flow_feature_df = (
    flow_wide
    .groupby("AssetCode", group_keys=False)
    .apply(add_flow_features)
    .reset_index(drop=True)
)

feature_df = price_feature_df.merge(
    flow_feature_df[
        [
            "Date", "AssetCode",
            "Foreign", "Institution",
            "Foreign_5D", "Foreign_20D",
            "Institution_5D", "Institution_20D",
            "TotalFlow", "TotalFlow_5D", "TotalFlow_20D",
            "Foreign_Positive_20D",
            "Institution_Positive_20D",
            "Flow_Confirm_20D"
        ]
    ],
    on=["Date", "AssetCode"],
    how="left"
)


# 상대강도 계산
# Date가 index와 column에 동시에 남는 문제를 피하기 위해
# wide table을 만들고, 각 pair별 결과를 list로 만든 뒤 concat 처리

close_wide = (
    price_df
    .pivot_table(index="Date", columns="AssetCode", values="Close", aggfunc="last")
    .sort_index()
)

relative_rows = []

relative_pair_specs = [
    ("005930", "KS11", "RS_vs_KOSPI"),
    ("000660", "KS11", "RS_vs_KOSPI"),
    ("042700", "KS11", "RS_vs_KOSPI"),
    ("SOXX", "^IXIC", "RS_vs_NASDAQ"),
    ("SMH", "^IXIC", "RS_vs_NASDAQ"),
    ("SOXX", "^GSPC", "RS_vs_SP500"),
    ("SMH", "^GSPC", "RS_vs_SP500"),
]

for asset, benchmark, rs_col in relative_pair_specs:
    if asset not in close_wide.columns:
        print(f"[WARN] 상대강도 계산 불가: asset 없음 - {asset}")
        continue

    if benchmark not in close_wide.columns:
        print(f"[WARN] 상대강도 계산 불가: benchmark 없음 - {benchmark}")
        continue

    rs = close_wide[asset] / close_wide[benchmark]

    temp = pd.DataFrame({
        "Date": close_wide.index,
        "AssetCode": asset,
        rs_col: rs.values,
        f"{rs_col}_Return_20D": rs.pct_change(20).values,
        f"{rs_col}_Return_60D": rs.pct_change(60).values,
    })

    relative_rows.append(temp.reset_index(drop=True))

if relative_rows:
    relative_df = pd.concat(relative_rows, axis=0, ignore_index=True)

    relative_df = (
        relative_df
        .groupby(["Date", "AssetCode"], as_index=False)
        .first()
    )

    feature_df = feature_df.merge(
        relative_df,
        on=["Date", "AssetCode"],
        how="left"
    )


def classify_price_flow(row):
    return_20d = row.get("Return_20D", np.nan)
    total_flow_20d = row.get("TotalFlow_20D", np.nan)

    if pd.isna(return_20d) or pd.isna(total_flow_20d):
        return np.nan

    price_up = return_20d > 0
    flow_up = total_flow_20d > 0

    if price_up and flow_up:
        return "Strong_Uptrend"

    if price_up and not flow_up:
        return "Weak_or_Fake_Rally"

    if not price_up and flow_up:
        return "Potential_Bottoming"

    return "Strong_Downtrend"


feature_df["Price_Flow_Regime"] = feature_df.apply(classify_price_flow, axis=1)

feature_df = feature_df.sort_values(["Date", "AssetCode"]).reset_index(drop=True)

feature_df.to_csv(OUT_FEATURE, index=False, encoding="utf-8-sig")

print("[DONE] Feature engineering completed")
print("feature:", OUT_FEATURE, feature_df.shape)

print("\n[FEATURE COUNT]")
feature_count = (
    feature_df.groupby(["AssetCode", "AssetName", "Market", "Source"])
    .size()
    .reset_index(name="RowCount")
    .sort_values(["Market", "AssetCode"])
    .reset_index(drop=True)
)
display(feature_count)

print("\n[FEATURE SAMPLE]")
display(feature_df.head(10))

print("\n[RECENT FEATURES]")
recent_cols = [
    "Date", "AssetCode", "AssetName", "Close",
    "Return_20D", "Return_60D", "MA30", "MA60", "MA120",
    "RSI14", "MACD", "MACD_Signal",
    "Foreign_20D", "Institution_20D", "TotalFlow_20D",
    "RS_vs_KOSPI_Return_20D",
    "RS_vs_NASDAQ_Return_20D",
    "RS_vs_SP500_Return_20D",
    "Price_Flow_Regime"
]
recent_cols = [c for c in recent_cols if c in feature_df.columns]

display(
    feature_df
    .sort_values(["AssetCode", "Date"])
    .groupby("AssetCode")
    .tail(3)[recent_cols]
    .reset_index(drop=True)
)

C:\Users\User\AppData\Local\Temp\ipykernel_19940\64028773.py:101: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  price_df
C:\Users\User\AppData\Local\Temp\ipykernel_19940\64028773.py:152: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  flow_wide
C:\Users\User\AppData\Local\Temp\ipykernel_19940\64028773.py:213: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed i

[DONE] Feature engineering completed
feature: market_agent_data\market_feature_data.csv (15738, 60)

[FEATURE COUNT]


,AssetCode,AssetName,Market,Source,RowCount
0,KRW=X,USD/KRW,FX,yfinance,1644
1,KQ11,KOSDAQ,KR_INDEX,FinanceDataReader,1550
2,KS11,KOSPI,KR_INDEX,FinanceDataReader,1550
3,000660,SK Hynix,KR_STOCK,FinanceDataReader,1550
4,005930,Samsung Electronics,KR_STOCK,FinanceDataReader,1550
5,042700,Hanmi Semiconductor,KR_STOCK,FinanceDataReader,1550
6,SMH,VanEck Semiconductor ETF,US_ETF,yfinance,1586
7,SOXX,iShares Semiconductor ETF,US_ETF,yfinance,1586
8,^GSPC,S&P 500,US_INDEX,yfinance,1586
9,^IXIC,NASDAQ Composite,US_INDEX,yfinance,1586



[FEATURE SAMPLE]


,Date,AssetCode,AssetName,Market,Source,Open,High,Low,Close,Volume,...,RS_vs_KOSPI,RS_vs_KOSPI_Return_20D,RS_vs_KOSPI_Return_60D,RS_vs_NASDAQ,RS_vs_NASDAQ_Return_20D,RS_vs_NASDAQ_Return_60D,RS_vs_SP500,RS_vs_SP500_Return_20D,RS_vs_SP500_Return_60D,Price_Flow_Regime
0,2020-01-01,KRW=X,USD/KRW,FX,yfinance,1154.400024,1154.400024,1145.449951,1153.750000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-02,000660,SK Hynix,KR_STOCK,FinanceDataReader,96000.000000,96200.000000,94100.000000,94700.000000,2342070,...,43.536827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-02,005930,Samsung Electronics,KR_STOCK,FinanceDataReader,55500.000000,56000.000000,55000.000000,55200.000000,12993228,...,25.377327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-02,042700,Hanmi Semiconductor,KR_STOCK,FinanceDataReader,4055.000000,4125.000000,3970.000000,3980.000000,366562,...,1.829742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-02,KQ11,KOSDAQ,KR_INDEX,FinanceDataReader,672.530000,674.300000,666.620000,674.020000,783730760,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2020-01-02,KRW=X,USD/KRW,FX,yfinance,1153.959961,1160.189941,1145.199951,1153.969971,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2020-01-02,KS11,KOSPI,KR_INDEX,FinanceDataReader,2201.210000,2202.320000,2171.840000,2175.170000,494677752,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2020-01-02,SMH,VanEck Semiconductor ETF,US_ETF,yfinance,71.894997,72.470001,71.654999,72.339996,5200400,...,NaN,NaN,NaN,0.007956,NaN,NaN,0.022205,NaN,NaN,NaN
8,2020-01-02,SOXX,iShares Semiconductor ETF,US_ETF,yfinance,84.753334,85.430000,84.333336,85.430000,1275300,...,NaN,NaN,NaN,0.009396,NaN,NaN,0.026223,NaN,NaN,NaN
9,2020-01-02,^GSPC,S&P 500,US_INDEX,yfinance,3244.669922,3258.139893,3235.530029,3257.850098,3459930000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



[RECENT FEATURES]


,Date,AssetCode,AssetName,Close,Return_20D,Return_60D,MA30,MA60,MA120,RSI14,MACD,MACD_Signal,Foreign_20D,Institution_20D,TotalFlow_20D,RS_vs_KOSPI_Return_20D,RS_vs_NASDAQ_Return_20D,RS_vs_SP500_Return_20D,Price_Flow_Regime
0,2026-04-22,000660,SK Hynix,1.223000e+06,0.229146,0.619868,9.981000e+05,948750.000000,780191.666667,87.861272,72722.437787,47717.967993,-2351698.0,1812913.0,-538785.0,0.080582,NaN,NaN,Weak_or_Fake_Rally
1,2026-04-23,000660,SK Hynix,1.225000e+06,0.312969,0.597132,1.007933e+06,956383.333333,786058.333333,86.736842,76583.036214,53490.981637,-1645624.0,2053493.0,407869.0,0.107107,NaN,NaN,Strong_Uptrend
2,2026-04-24,000660,SK Hynix,1.222000e+06,0.325380,0.660326,1.018333e+06,964483.333333,791591.666667,85.897436,78495.664198,58491.918149,-1054921.0,1909480.0,854559.0,0.113184,NaN,NaN,Strong_Uptrend
3,2026-04-22,005930,Samsung Electronics,2.175000e+05,0.150794,0.428102,1.966000e+05,186993.333333,150487.500000,77.496484,8563.683704,6856.243116,-31923416.0,3432491.0,-28490925.0,0.011700,NaN,NaN,Weak_or_Fake_Rally
4,2026-04-23,005930,Samsung Electronics,2.245000e+05,0.246530,0.476003,1.978200e+05,188200.000000,151529.166667,77.240398,9133.144854,7311.623464,-17482155.0,5776657.0,-11705498.0,0.051085,NaN,NaN,Weak_or_Fake_Rally
5,2026-04-24,005930,Samsung Electronics,2.195000e+05,0.221480,0.443130,1.990200e+05,189323.333333,152520.833333,69.298246,9076.362052,7664.571181,-12135262.0,5034946.0,-7100316.0,0.025919,NaN,NaN,Weak_or_Fake_Rally
6,2026-04-22,042700,Hanmi Semiconductor,2.935000e+05,-0.021667,0.722418,2.855500e+05,256875.000000,197576.666667,70.270270,5728.127602,4666.630636,-1000662.0,245954.0,-754708.0,-0.139915,NaN,NaN,Strong_Downtrend
7,2026-04-23,042700,Hanmi Semiconductor,2.935000e+05,0.055755,0.715371,2.847667e+05,258915.000000,198830.000000,68.926554,5901.228585,4913.550226,-944828.0,229261.0,-715567.0,-0.109778,NaN,NaN,Weak_or_Fake_Rally
8,2026-04-24,042700,Hanmi Semiconductor,2.955000e+05,0.072595,0.743363,2.844333e+05,261015.000000,200140.833333,76.363636,6129.142705,5156.668722,-752460.0,207002.0,-545458.0,-0.099129,NaN,NaN,Weak_or_Fake_Rally
9,2026-04-22,KQ11,KOSDAQ,1.181120e+03,0.018602,0.217210,1.122782e+03,1123.077167,1021.037750,80.210149,16.695040,5.502192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
